# Feature Hierarchy Explorer: a hands-on review

This notebook is a human-oriented tour of the explorer in `bea-tools`. It uses a small synthetic clinical-style table so every count can be checked by eye. Nothing here assumes real patient data.

The explorer deliberately separates four questions:

1. **Independent levels (S1):** What values occur in each column?
2. **Nested census (S2):** Which observed paths occur in an ordered hierarchy?
3. **Grain evidence (S3):** Do supplied keys determine other columns in these observed rows?
4. **Pairs and absence:** How do dimensions map to one another, and which domain cells were not observed?

> Important: unobserved does not mean impossible, and an exact functional dependency in a sample is evidence—not proof of real-world semantics.

## 1. Setup and a reviewable dataset

The table includes repeated exams, two sites, two modalities, left/right sides, a missing finding, and one deliberate conflict for `(exam_id, side) = (E001, L)`.

In [ ]:
import json

import pandas as pd

import bea_tools
from bea_tools import (
    KeySpec,
    census,
    explore,
    grain,
    infer_schema,
    levels,
    render_plaintext,
)

pd.set_option("display.max_columns", None)

df = pd.DataFrame(
    {
        "exam_id": ["E001", "E001", "E001", "E002", "E002", "E002", "E003", "E003", "E004", "E004"],
        "site": ["North", "North", "North", "North", "North", "North", "South", "South", "South", "South"],
        "modality": ["MRI", "MRI", "MRI", "CT", "CT", "CT", "MRI", "MRI", "CT", "CT"],
        "side": ["L", "L", "R", "L", "L", "R", "L", "R", "L", "R"],
        "finding": ["clear", "scar", "clear", "clear", "clear", "scar", "nodule", "nodule", "clear", None],
        "severity": [0, 1, 0, 0, 0, 1, 2, 2, 0, pd.NA],
    }
)

df

Before using the combined explorer, we will call each analytical surface independently. This is useful when you only need one answer and want the smallest result.

## 2. S1: independent level counts

`levels()` counts every feature on its own population. A top-N limit for one feature never filters rows for another feature. Missing values count as a typed level unless `dropna=True`.

In [ ]:
independent = levels(
    df,
    features=["site", "modality", "side", "finding"],
    top_n=3,
)
print(render_plaintext(independent, width=88, max_lines=40))

In [ ]:
# Population accounting is explicit for each feature.
pd.DataFrame(
    {
        "feature": [entry["column"]["value"] for entry in independent["per_feature"]],
        "levels_total": [entry["levels_total"] for entry in independent["per_feature"]],
        "levels_reported": [entry["levels_reported"] for entry in independent["per_feature"]],
        "reported_rows": [entry["reported_rows"] for entry in independent["per_feature"]],
        "unreported_rows": [entry["unreported_rows"] for entry in independent["per_feature"]],
    }
)

**Review checkpoint:** `finding` has four typed levels: `clear`, `scar`, `nodule`, and missing. Because `top_n=3`, one level is omitted from display, but the omitted row mass remains explicit.

## 3. S2: a bounded nested census

`census()` treats dimension order as meaningful. Here we ask for observed `site → modality → side → finding` prefixes. The default `top_n_mode="post"` preserves population counts while collapsing omitted branches. The plaintext view prints each parent with its descendants before moving to the next sibling, and names the dimension on every node. Root totals and omitted row counts remain visible even in totals-only requests.

In [ ]:
nested = census(
    df,
    dimensions=["site", "modality", "side", "finding"],
    top_n=2,
    top_n_per_parent=True,
    max_nodes=40,
)
print(render_plaintext(nested, width=88, max_lines=60))

In [ ]:
# Every expanded parent conserves row mass.
nodes = {node["node_id"]: node for node in nested["tree"]["nodes"]}
nodes["root"] = nested["tree"]["root"]
checks = []
for parent in nodes.values():
    if parent["expansion_state"] != "expanded":
        continue
    child_total = sum(
        node["count"] for node in nodes.values() if node.get("parent_id") == parent["node_id"]
    )
    checks.append(
        {
            "parent": parent["node_id"],
            "parent_count": parent["count"],
            "emitted_children": child_total,
            "omitted_child_rows": parent["omitted_child_rows"],
            "conserved": parent["count"] == child_total + parent["omitted_child_rows"],
        }
    )
pd.DataFrame(checks)

### Conditional pre-selection

Pre mode is intentionally different: it selects retained levels first, intersects them into one common cohort, and recomputes every reported prefix on that cohort. The result is marked conditional.

In [ ]:
conditional = census(
    df,
    dimensions=["site", "modality", "side"],
    top_n=1,
    top_n_mode="pre",
)
print(render_plaintext(conditional, width=88, max_lines=30))
pd.DataFrame(conditional["scopes"])

**Review checkpoint:** compare `input_rows`, `restriction_excluded_rows`, and `evaluated_rows`. Output limits do not silently change these population definitions.

## 4. S3: explicit keys and functional-dependency evidence

Keys are always supplied by the caller. A composite key uses `KeySpec`, which avoids ambiguity with a dataframe column whose label is itself a tuple.

A violating group has more than one observed target value. `affected_rows` counts every row in those groups; it does not claim which row is erroneous.

In [ ]:
grain_result = grain(
    df,
    candidate_keys=[
        "exam_id",
        KeySpec("exam_side", ("exam_id", "side")),
    ],
)

def typed_label(record):
    return record.get("value", f"<{record['type']}>")

dependency_table = pd.DataFrame(
    {
        "key": [item["key_name"] for item in grain_result["dependencies"]],
        "target": [typed_label(item["target"]) for item in grain_result["dependencies"]],
        "holds": [item["holds"] for item in grain_result["dependencies"]],
        "groups": [item["evaluated_groups"] for item in grain_result["dependencies"]],
        "violating_groups": [item["violating_groups"] for item in grain_result["dependencies"]],
        "affected_rows": [item["affected_rows"] for item in grain_result["dependencies"]],
        "singleton_groups": [item["singleton_groups"] for item in grain_result["dependencies"]],
    }
)
dependency_table

In [ ]:
# The deliberate E001/L conflict should appear here.
dependency_table.query("key == 'exam_side' and target == 'finding'")

## 5. Schema proposals are suggestions, not hidden selections

`infer_schema()` reports reviewable cardinality, dtype, missingness, name hints, and optional FD evidence. It does not choose the dimensions or keys for a later analysis.

In [ ]:
proposal = infer_schema(df, candidate_keys=["exam_id"])
pd.DataFrame(
    {
        "column": [typed_label(item["column"]) for item in proposal["proposals"]],
        "proposed_role": [item["proposed_role"] for item in proposal["proposals"]],
        "fd_status": [
            item["fd_evidence"]["status"]
            if isinstance(item["fd_evidence"], dict)
            else item["fd_evidence"]
            for item in proposal["proposals"]
        ],
    }
)

## 6. Combined exploration, pair relationships, and absence

`explore()` orchestrates the independent surfaces. Here we also declare reference-domain levels that are absent globally (`West`, `PET`, and `U`). Absence totals are exact, while examples are bounded. The plaintext view names the pair columns, states the context and evaluated population, and separates the absence classes. We allow 160 lines here so the combined evidence fits; render an individual section when you need a shorter report. None of the absence classes claim structural impossibility.

In [ ]:
combined = explore(
    df,
    dimensions=["site", "modality", "side"],
    features=["site", "modality", "side", "finding"],
    candidate_keys=["exam_id", KeySpec("exam_side", ("exam_id", "side"))],
    include_pairs=True,
    include_absence=True,
    reference_domains={
        "site": ["North", "South", "West"],
        "modality": ["CT", "MRI", "PET"],
        "side": ["L", "R", "U"],
    },
    max_absence_cells=8,
)
print(render_plaintext(combined, width=100, max_lines=160))

### Topology-only context for external models

`detail="topology"` retains labels, hierarchy paths, and qualitative relationships while suppressing counts, population sizes, cardinalities, support totals, and association scores. It canonically sorts displayed siblings so frequency-ranked output order does not disclose relative prevalence. This is a presentation filter rather than de-identification: review categorical values, paths, contexts, relationships, and any frequency-based `top_n` selection before sharing the rendered string. Do not share `combined` or `combined.to_dict()` when its quantitative fields must remain internal.

In [ ]:
external_context = render_plaintext(
    combined,
    detail="topology",
    width=100,
    max_lines=160,
)
print(external_context)

In [ ]:
pair_table = pd.DataFrame(
    [
        {
            "pair": " / ".join(str(typed_label(column)) for column in item["columns"]),
            "evaluated_rows": item["evaluated_rows"],
            "relation": item["relation"],
            "cramers_v": item["cramers_v"],
            "observed_cells": item["observed_cells"],
            "absent_cells": item["absence"]["absent_cells"],
            "zero_support_absence": item["absence"]["classes"]["unobserved_zero_support"],
            "bounded_examples": len(item["absence"]["examples"]),
        }
        for item in combined["sections"]["pairs"]["pairs"]
    ]
)
pair_table

**How to read relation orientation:** dimensions follow the supplied order. `1:n` means each B maps to one A while at least one A maps to multiple Bs. Pairwise saturation still cannot rule out a higher-order constraint such as XOR.

## 7. Strict JSON and compact parent-linked nodes

Every public result uses standard-library data and schema version `0.2`. Tree nodes store parent and level references rather than repeating full paths.

In [ ]:
strict_json = json.dumps(combined.to_dict(), allow_nan=False, sort_keys=True)
print(f"schema={combined['schema_version']}, JSON bytes={len(strict_json.encode('utf-8')):,}")

pd.DataFrame(combined["sections"]["census"]["tree"]["nodes"]).head(8)

## 8. Functional API and `DataFrame.bea` are equivalent

The accessor is intentionally thin; it delegates to the same implementation rather than maintaining a second engine.

In [ ]:
functional = levels(df, ["site", "side"], top_n=2).to_dict()
accessor = df.bea.levels(["site", "side"], top_n=2).to_dict()
assert functional == accessor
print("Functional and accessor results are identical.")

## 9. Safe rendering of terminal-sensitive labels

Safe mode is ASCII-only and escapes non-ASCII and terminal controls. Native display mode is available with `bea-tools[unicode]`. All string values are quoted: the literal string `<NA>` remains visibly distinct from a missing value, and strings such as `"1"` remain distinct from numbers. Line truncation is explicitly marked only when more content exists.

In [ ]:
adversarial = pd.DataFrame(
    {"label": [None, "<NA>", "snowman ☃", "line\nbreak", "\x1b[31mred"]}
)
safe_text = render_plaintext(levels(adversarial), width=42, max_lines=12)
print(safe_text)
assert "\x1b" not in safe_text

## Suggested review exercises

Try these edits and rerun the relevant cells:

- Change `top_n_per_parent=True` to `False` and compare the retained tree.
- Set `dropna=True` for `levels()` and `grain()` and inspect their separate scopes.
- Remove the conflicting `E001/L` row and confirm that `exam_side → finding` holds.
- Set `max_nodes=0` and inspect totals-only omission accounting.
- Add a `West` row without updating `reference_domains`; the explorer should reject the mismatched declared domain rather than discard the observation.
- Duplicate the whole dataframe and confirm that unpruned counts double while exact FD truth values do not change.

For larger data, remember that exact top-N still scans the eligible rows. Output limits bound representation and deeper expansion; they do not make exact counting sublinear.